In [1]:

from __future__ import annotations

import csv
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import textwrap
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    MATPLOTLIB_ERROR = repr(exc)
else:
    MATPLOTLIB_ERROR = None


# ============================================================
# P2 Integrated Engineering Blueprint - one executable cell
# 목적:
#   1) 오늘 생성된 P2_2/P2_3/P2_4/P2_5/P2_6 대표 notebooks를 복제한다.
#   2) 핵심 데이터와 산출물의 shape/SHA/key/sample/status를 재검산한다.
#   3) 기존 산출물에 저장된 P5/P6 수치만 읽어 해석한다. 여기서 모델을 새로 학습하지 않는다.
#   4) M/W13식 V00~Final 단계형 blueprint로 검토자가 따라갈 수 있는 engineering handoff를 만든다.
# ============================================================

RANDOM_STATE = 3085
PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")
OUT_ROOT = PROJECT_ROOT / "workbook/p2/p2_integrated_engineering_blueprint_v1"
GIT_SHORT = "unknown"
try:
    GIT_SHORT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
except Exception:
    pass
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + f"_{GIT_SHORT}"
RUN_DIR = OUT_ROOT / "runs" / RUN_ID
CLONE_DIR = RUN_DIR / "cloned_notebooks"
QA_DIR = RUN_DIR / "qa"
REPORT_DIR = RUN_DIR / "reports"
LOG_DIR = RUN_DIR / "logs"
FIG_DIR = RUN_DIR / "figures"
ART_DIR = RUN_DIR / "artifacts"
for directory in [CLONE_DIR, QA_DIR, REPORT_DIR, LOG_DIR, FIG_DIR, ART_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def run_cmd(args: list[str]) -> tuple[int, str]:
    try:
        out = subprocess.check_output(args, cwd=PROJECT_ROOT, stderr=subprocess.STDOUT, text=True)
        return 0, out.strip()
    except subprocess.CalledProcessError as exc:
        return exc.returncode, exc.output.strip()
    except Exception as exc:
        return 99, repr(exc)


def read_csv_safe(path: Path) -> pd.DataFrame:
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, low_memory=False)


def load_table(path: Path) -> tuple[pd.DataFrame | None, str, str | None]:
    if not path.exists():
        return None, "missing", None
    suffix = path.suffix.lower()
    try:
        if suffix == ".parquet":
            return pd.read_parquet(path), "parquet", None
        if suffix == ".csv":
            return read_csv_safe(path), "csv", None
        if suffix == ".json":
            with path.open(encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, dict):
                return pd.json_normalize(data), "json_normalize", None
            if isinstance(data, list):
                return pd.json_normalize(data), "json_normalize", None
            return pd.DataFrame({"value": [data]}), "json_scalar", None
        if suffix in {".xlsx", ".xls"}:
            xls = pd.ExcelFile(path)
            rows = []
            for sheet in xls.sheet_names:
                df = pd.read_excel(path, sheet_name=sheet)
                rows.append({"sheet": sheet, "rows": len(df), "columns": len(df.columns)})
            return pd.DataFrame(rows), "excel_sheet_inventory", None
    except Exception as exc:
        return None, "load_error", repr(exc)
    return None, "unprofiled", None


def shape_repr(path: Path) -> tuple[str, str, str | None]:
    df, loader, err = load_table(path)
    if df is None:
        return "", loader, err
    return f"{len(df)} x {len(df.columns)}", loader, err


def notebook_meta(path: Path) -> dict[str, Any]:
    meta = {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
        "sha256": sha256_file(path),
        "cell_n": None,
        "code_cell_n": None,
        "markdown_cell_n": None,
        "executed_code_cell_n": None,
        "error_output_n": None,
        "stored_output_n": None,
        "parse_status": "MISSING" if not path.exists() else "UNREAD",
    }
    if not path.exists():
        return meta
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
        cells = nb.get("cells", [])
        meta.update(
            {
                "cell_n": len(cells),
                "code_cell_n": sum(1 for c in cells if c.get("cell_type") == "code"),
                "markdown_cell_n": sum(1 for c in cells if c.get("cell_type") == "markdown"),
                "executed_code_cell_n": sum(1 for c in cells if c.get("cell_type") == "code" and c.get("execution_count") is not None),
                "error_output_n": sum(1 for c in cells for o in c.get("outputs", []) if o.get("output_type") == "error"),
                "stored_output_n": sum(len(c.get("outputs", [])) for c in cells if c.get("cell_type") == "code"),
                "parse_status": "PASS",
            }
        )
    except Exception as exc:
        meta["parse_status"] = f"FAIL:{type(exc).__name__}:{exc}"
    return meta


def atomic_csv(df: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
    tmp.replace(path)


def atomic_text(text: str, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def md_table(df: pd.DataFrame, cols: list[str] | None = None, max_rows: int | None = None) -> str:
    if cols is None:
        cols = list(df.columns)
    use = df[cols].copy() if len(df) else pd.DataFrame(columns=cols)
    if max_rows is not None:
        use = use.head(max_rows)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in use.iterrows():
        vals = []
        for col in cols:
            val = row.get(col, "")
            if isinstance(val, float):
                txt = "" if pd.isna(val) else f"{val:.6g}"
            else:
                txt = "" if pd.isna(val) else str(val)
            vals.append(txt.replace("\n", " ").replace("|", "\\|"))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def pct_range_violations(df: pd.DataFrame) -> int:
    n = 0
    for col in [c for c in df.columns if c.endswith("_pct") or c.endswith("비율")]:
        vals = pd.to_numeric(df[col], errors="coerce")
        n += int(((vals < 0) | (vals > 100)).sum())
    return n


def parse_bootstrap_ci(report_path: Path) -> str:
    if not report_path.exists():
        return ""
    text = report_path.read_text(encoding="utf-8")
    match = re.search(r"bootstrap residual D 95% CI:\s*`\[([^\]]+)\]%", text)
    return match.group(1) if match else ""


git_code, git_full = run_cmd(["git", "rev-parse", "HEAD"])
_, git_status = run_cmd(["git", "status", "--short"])
dirty = bool(git_status.strip())
env = {
    "run_id": RUN_ID,
    "utc_started_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "output_run_dir": str(RUN_DIR),
    "git_commit_full": git_full if git_code == 0 else "",
    "git_commit_short": GIT_SHORT,
    "git_dirty": dirty,
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib_error": MATPLOTLIB_ERROR,
    "random_state": RANDOM_STATE,
}
atomic_text(json.dumps(env, ensure_ascii=False, indent=2), LOG_DIR / "execution_environment.json")


representative_notebooks = [
    {"stage": "P2_2", "label": "P2_G0_source_eda", "path": "workbook/p2/p2_2/final/eda/P2_G0.ipynb", "reason": "초기 원천 EDA와 파일 구조 확인"},
    {"stage": "P2_2", "label": "P2_G1_kedi_structure", "path": "workbook/p2/p2_2/final/eda/P2_G1_kedi.ipynb", "reason": "KEDI 고등교육 구조자료 검사"},
    {"stage": "P2_2", "label": "P2_G1_concat_outcome", "path": "workbook/p2/p2_2/final/eda/P2_G1_concat_eda.ipynb", "reason": "성적·입결·취업·진학 outcome spine 결합"},
    {"stage": "P2_2", "label": "P2_G2_visual_admission", "path": "workbook/p2/p2_2/final/eda/P2_G2_정시입결_visual_eda.ipynb", "reason": "입결 proxy와 정시 입결 시각 검사"},
    {"stage": "P2_3", "label": "P3_1_handoff_builder", "path": "workbook/p2/p2_3/p3_1.ipynb", "reason": "D01-D05와 D08 handoff 구성"},
    {"stage": "P2_3", "label": "P3_2_goms_context", "path": "workbook/p2/p2_3/p3_2.ipynb", "reason": "GOMS D06/D07 노동시장 context 구성"},
    {"stage": "P2_4", "label": "P4_source_parquet_csv_eda", "path": "workbook/p2/p2_4/p2_G4_source_parquet_csv_eda.ipynb", "reason": "handoff parquet/csv 전체 source inspection"},
    {"stage": "P2_4", "label": "P4_dataset_cell_inspection", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/p2_G4_dataset_cell_inspection.ipynb", "reason": "dataset-by-dataset inspection"},
    {"stage": "P2_4", "label": "P4_contract_cleaning_1", "path": "workbook/p2/p2_4/p2_G4_1.ipynb", "reason": "P4 계약·무결성 검증"},
    {"stage": "P2_4", "label": "P4_contract_cleaning_2", "path": "workbook/p2/p2_4/p2_G4_2.ipynb", "reason": "P4 readiness 후속 검증"},
    {"stage": "P2_4", "label": "P2_grade_formation_strict", "path": "workbook/p2/p2_4/p2_grade_formation_v1_strict/p2_grade_formation_strict.ipynb", "reason": "strict P2 grade formation branch"},
    {"stage": "P2_4", "label": "P3_grade_residual_strict", "path": "workbook/p2/p2_4/p3_grade_residual_v1_strict/p3_grade_residual_strict.ipynb", "reason": "strict P3 residual handoff"},
    {"stage": "P2_4", "label": "P4_grade_signal_outcomes_strict", "path": "workbook/p2/p2_4/p4_grade_signal_outcomes_v1_strict/p4_grade_signal_outcomes_strict.ipynb", "reason": "strict P4 grade-signal/outcome analysis"},
    {"stage": "P2_5", "label": "P5_major7_heterogeneity_strict", "path": "workbook/p2/p2_5/p2_g5_1_strict.ipynb", "reason": "P5 major7 heterogeneity strict result"},
    {"stage": "P2_5", "label": "P5_A2_visual_summary", "path": "workbook/p2/p2_5/P2_G5_A2.ipynb", "reason": "P5 visual/numeric summary"},
    {"stage": "P2_6", "label": "P6_confirmatory_closure", "path": "workbook/p2/P2_6/p4_confirmatory_closure.ipynb", "reason": "P4 confirmatory closure"},
    {"stage": "P2_6", "label": "P6_strict_chain_runup", "path": "workbook/p2/P2_6/P2_G6_1.ipynb", "reason": "P3->P4->P5/P6 strict chain run-up"},
]

notebook_rows = []
for item in representative_notebooks:
    src = PROJECT_ROOT / item["path"]
    meta = notebook_meta(src)
    clone_rel = Path(item["path"])
    clone_path = CLONE_DIR / clone_rel
    clone_path.parent.mkdir(parents=True, exist_ok=True)
    clone_status = "MISSING"
    if src.exists():
        shutil.copy2(src, clone_path)
        clone_status = "COPIED"
    row = {**item, **meta}
    row["clone_path"] = str(clone_path)
    row["clone_status"] = clone_status
    row["clone_sha256"] = sha256_file(clone_path) if clone_path.exists() else None
    row["clone_hash_match"] = row["sha256"] == row["clone_sha256"] if row["sha256"] else False
    notebook_rows.append(row)
notebook_inventory = pd.DataFrame(notebook_rows)
atomic_csv(notebook_inventory, QA_DIR / "representative_notebook_clone_inventory.csv")


key_files = [
    {"label": "D08_active_mart", "path": "workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet", "role": "active 2024 151-column mart"},
    {"label": "D08_clean_integrity_mart", "path": "workbook/p2/p2_4/p4_preprocessing_integrity_v1/data/clean_department_model_base_2024.parquet", "role": "preprocessing/integrity clean mart"},
    {"label": "D08_strict_drop_mart", "path": "workbook/p2/p2_4/source_eda/strict_clean_v1/mart_department_model_base_2024_strict_drop.parquet", "role": "strict model input mart"},
    {"label": "source_catalog", "path": "workbook/p2/p2_4/p4_preprocessing_integrity_v1/qa/data_source_catalog.csv", "role": "source provenance catalog"},
    {"label": "source_file_inventory", "path": "workbook/p2/p2_4/p4_preprocessing_integrity_v1/qa/data_source_file_inventory.csv", "role": "source file inventory"},
    {"label": "column_registry_v4", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_registry_v4.csv", "role": "151-column role contract"},
    {"label": "phase_sample_registry_final", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv", "role": "phase sample registry"},
    {"label": "phase_sample_membership_final", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet", "role": "row-level sample masks"},
    {"label": "dim_school_split", "path": "workbook/p2/p2_3/p4_handoff_candidate/shared/dim_school_split.csv", "role": "school split"},
    {"label": "p4_final_modeling_readiness", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/reports/P4_FINAL_MODELING_READINESS.json", "role": "P4 readiness status"},
    {"label": "preprocessing_manifest", "path": "workbook/p2/p2_4/p4_preprocessing_integrity_v1/PREPROCESSING_INTEGRITY_MANIFEST.json", "role": "preprocessing integrity manifest"},
    {"label": "p5_strict_status", "path": "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/reports/P5_2024_MAJOR7_HETEROGENEITY_V2_STRICT_STATUS.json", "role": "P5 strict status"},
    {"label": "p5_strict_report", "path": "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/reports/P5_2024_MAJOR7_HETEROGENEITY_V2_STRICT_REPORT.md", "role": "P5 strict report"},
    {"label": "p5_strict_insight", "path": "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/artifacts/P5_STRICT_INSIGHT_SUMMARY.csv", "role": "P5 AME summary"},
    {"label": "p5_difference", "path": "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/artifacts/P5_TABLE3_PROGRESSION_MINUS_EMPLOYMENT_AME.csv", "role": "P5 progression-employment differences"},
    {"label": "p6_status", "path": "workbook/p2/P2_6/qa/P2_G6_1_STATUS_TABLE.csv", "role": "P6 status"},
    {"label": "p6_key_numbers", "path": "workbook/p2/P2_6/artifacts/P2_G6_1_KEY_NUMBERS.csv", "role": "P6 key numbers"},
    {"label": "p6_report", "path": "workbook/p2/P2_6/reports/P2_G6_1_STRICT_CHAIN_RUNUP_REPORT.md", "role": "P6 strict chain report"},
]

file_rows = []
for item in key_files:
    path = PROJECT_ROOT / item["path"]
    shape, loader, err = shape_repr(path)
    file_rows.append(
        {
            **item,
            "absolute_path": str(path),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else None,
            "shape": shape,
            "loader": loader,
            "load_error": err,
            "sha256": sha256_file(path),
        }
    )
file_inventory = pd.DataFrame(file_rows)
atomic_csv(file_inventory, QA_DIR / "key_file_inventory.csv")


# ----------------------------
# Integrity checks
# ----------------------------
d08_path = PROJECT_ROOT / "workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet"
d08 = pd.read_parquet(d08_path)
strict_path = PROJECT_ROOT / "workbook/p2/p2_4/source_eda/strict_clean_v1/mart_department_model_base_2024_strict_drop.parquet"
strict = pd.read_parquet(strict_path) if strict_path.exists() else pd.DataFrame()
clean_path = PROJECT_ROOT / "workbook/p2/p2_4/p4_preprocessing_integrity_v1/data/clean_department_model_base_2024.parquet"
clean = pd.read_parquet(clean_path) if clean_path.exists() else pd.DataFrame()

target_cols = ["a_rate_pct", "health_employment_rate_pct", "graduate_school_progression_rate_pct"]
key_audit_rows = []
for name, df, path in [
    ("D08_active", d08, d08_path),
    ("D08_clean_integrity", clean, clean_path),
    ("D08_strict_drop", strict, strict_path),
]:
    if df.empty:
        continue
    row = {
        "dataset": name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "shape": f"{len(df)} x {len(df.columns)}",
        "sha256": sha256_file(path),
        "outcome_row_id_exists": "outcome_row_id" in df.columns,
        "outcome_row_id_null_n": int(df["outcome_row_id"].isna().sum()) if "outcome_row_id" in df.columns else None,
        "outcome_row_id_duplicate_rows": int(df.duplicated("outcome_row_id").sum()) if "outcome_row_id" in df.columns else None,
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "school_n": int(df["school_uid"].nunique()) if "school_uid" in df.columns else None,
        "campus_n": int(df["campus_uid"].nunique()) if "campus_uid" in df.columns else None,
        "dept_n": int(df["dept_uid"].nunique()) if "dept_uid" in df.columns else None,
        "major7_non_null_n": int(df["major_group_7"].notna().sum()) if "major_group_7" in df.columns else None,
        "major7_missing_n": int(df["major_group_7"].isna().sum()) if "major_group_7" in df.columns else None,
        "pct_range_violation_n": pct_range_violations(df),
    }
    for t in target_cols:
        row[f"{t}_non_null_n"] = int(df[t].notna().sum()) if t in df.columns else None
    key_audit_rows.append(row)
key_audit = pd.DataFrame(key_audit_rows)
atomic_csv(key_audit, QA_DIR / "key_integrity_audit.csv")


sample_path = PROJECT_ROOT / "workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv"
sample_summary = read_csv_safe(sample_path) if sample_path.exists() else pd.DataFrame()
if not sample_summary.empty:
    atomic_csv(sample_summary, QA_DIR / "phase_sample_summary.csv")

split_path = PROJECT_ROOT / "workbook/p2/p2_3/p4_handoff_candidate/shared/dim_school_split.csv"
split_df = read_csv_safe(split_path) if split_path.exists() else pd.DataFrame()
split_summary = pd.DataFrame()
if not split_df.empty and "split" in split_df.columns:
    split_summary = split_df["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="school_n")
    atomic_csv(split_summary, QA_DIR / "school_split_summary.csv")

target_rows = []
for df_name, df in [("D08_active", d08), ("D08_strict_drop", strict)]:
    for target in target_cols:
        if target not in df.columns:
            continue
        vals = pd.to_numeric(df[target], errors="coerce")
        target_rows.append(
            {
                "dataset": df_name,
                "target": target,
                "n": int(vals.notna().sum()),
                "missing_n": int(vals.isna().sum()),
                "mean": float(vals.mean()) if vals.notna().any() else np.nan,
                "std": float(vals.std()) if vals.notna().sum() > 1 else np.nan,
                "min": float(vals.min()) if vals.notna().any() else np.nan,
                "p25": float(vals.quantile(0.25)) if vals.notna().any() else np.nan,
                "median": float(vals.median()) if vals.notna().any() else np.nan,
                "p75": float(vals.quantile(0.75)) if vals.notna().any() else np.nan,
                "max": float(vals.max()) if vals.notna().any() else np.nan,
                "zero_n": int((vals == 0).sum()),
                "hundred_n": int((vals == 100).sum()),
            }
        )
target_profile = pd.DataFrame(target_rows)
atomic_csv(target_profile, QA_DIR / "target_profile_recomputed.csv")


# ----------------------------
# Existing numeric result readout. This cell reads, it does not refit.
# ----------------------------
p6_numbers_path = PROJECT_ROOT / "workbook/p2/P2_6/artifacts/P2_G6_1_KEY_NUMBERS.csv"
p6_status_path = PROJECT_ROOT / "workbook/p2/P2_6/qa/P2_G6_1_STATUS_TABLE.csv"
p6_report_path = PROJECT_ROOT / "workbook/p2/P2_6/reports/P2_G6_1_STRICT_CHAIN_RUNUP_REPORT.md"
p5_status_path = PROJECT_ROOT / "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/reports/P5_2024_MAJOR7_HETEROGENEITY_V2_STRICT_STATUS.json"
p5_insight_path = PROJECT_ROOT / "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/artifacts/P5_STRICT_INSIGHT_SUMMARY.csv"
p5_diff_path = PROJECT_ROOT / "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/artifacts/P5_TABLE3_PROGRESSION_MINUS_EMPLOYMENT_AME.csv"

p6_numbers = read_csv_safe(p6_numbers_path) if p6_numbers_path.exists() else pd.DataFrame(columns=["metric", "value", "unit"])
p6_status = read_csv_safe(p6_status_path) if p6_status_path.exists() else pd.DataFrame(columns=["status_key", "status"])
p5_insight = read_csv_safe(p5_insight_path) if p5_insight_path.exists() else pd.DataFrame()
p5_diff = read_csv_safe(p5_diff_path) if p5_diff_path.exists() else pd.DataFrame()
with p5_status_path.open(encoding="utf-8") as f:
    p5_status_json = json.load(f) if p5_status_path.exists() else {}

p6_metric = {str(r.metric): float(r.value) for r in p6_numbers.itertuples(index=False) if pd.notna(r.value)}
bootstrap_ci = parse_bootstrap_ci(p6_report_path)
result_rows = []
for metric, value in p6_metric.items():
    result_rows.append({"source": "P2_6", "metric": metric, "value": value, "unit": p6_numbers.loc[p6_numbers["metric"].eq(metric), "unit"].iloc[0]})
if not p5_diff.empty:
    for _, r in p5_diff.iterrows():
        result_rows.append(
            {
                "source": "P2_5",
                "metric": f"P5_DIFF_{r['major_group_7']}",
                "value": float(r["difference"]),
                "unit": "probability-scale AME difference; multiply by 100 for percentage points",
            }
        )
result_summary = pd.DataFrame(result_rows)
atomic_csv(result_summary, QA_DIR / "existing_result_numeric_readout.csv")

interpretation_rows = [
    {
        "topic": "P3-S grade residual",
        "observation": f"OOF R2={p6_metric.get('P3_FULL_OOF_R2', np.nan):.3f}, locked test R2={p6_metric.get('P3_FULL_TEST_R2', np.nan):.3f}, OOF MAE={p6_metric.get('P3_FULL_OOF_MAE', np.nan):.3f}",
        "interpretation": "구조+정책 기반 A비율 예측력은 제한적이다. residual은 생성됐지만 강한 예측 성능 산출물로 해석하지 않는다.",
        "limit": "기존 P2_6 artifact readout이며 이 notebook에서 Ridge/OLS를 재적합하지 않았다.",
    },
    {
        "topic": "P4 raw A signal",
        "observation": f"RAW_A employment AME={p6_metric.get('P4_RAW_A_EMPLOYMENT_AME_PP', np.nan):.3f}pp, progression AME={p6_metric.get('P4_RAW_A_PROGRESSION_AME_PP', np.nan):.3f}pp",
        "interpretation": "기존 confirmatory artifact 기준으로 A비율 10pp 관련 신호는 대학원 진학 쪽 수치가 건강보험 취업보다 크다.",
        "limit": "인과효과가 아니라 조건부 association readout이다.",
    },
    {
        "topic": "P4 residual signal",
        "observation": f"OOF residual employment AME={p6_metric.get('P4_RESID_EMPLOYMENT_AME_PP', np.nan):.3f}pp, progression AME={p6_metric.get('P4_RESID_PROGRESSION_AME_PP', np.nan):.3f}pp, D={p6_metric.get('P4_RESID_D_PP', np.nan):.3f}pp, CI=[{bootstrap_ci}]pp",
        "interpretation": "OOF residual readout에서도 진학 쪽 추가 신호가 더 크게 저장돼 있다.",
        "limit": "P2_6 보고서가 fixed encoder/fast least-squares bootstrap approximation 경고를 둔다.",
    },
]
if not p5_diff.empty:
    top_prog = p5_diff.assign(diff_pp=lambda x: x["difference"].astype(float) * 100).sort_values("diff_pp", ascending=False).head(3)
    interpretation_rows.append(
        {
            "topic": "P5 major7 heterogeneity",
            "observation": "; ".join(f"{r.major_group_7}: progression-employment {r.diff_pp:.2f}pp" for r in top_prog.itertuples(index=False)),
            "interpretation": "strict P5 artifact에서는 NAT, ENG, ART/MED 순으로 진학-취업 차이가 큰 쪽으로 관찰된다.",
            "limit": "계열별 탐색 결과이며 context가 원인이라는 해석은 금지한다.",
        }
    )
interpretation = pd.DataFrame(interpretation_rows)
atomic_csv(interpretation, QA_DIR / "numeric_interpretation_scaffold.csv")


# ----------------------------
# Issue register and status matrix
# ----------------------------
issue_rows = []

def add_issue(issue_id: str, severity: str, evidence: str, impact: str, decision: str) -> None:
    issue_rows.append(
        {
            "issue_id": issue_id,
            "severity": severity,
            "evidence": evidence,
            "impact": impact,
            "required_decision": decision,
        }
    )

if "split" not in d08.columns:
    add_issue("WARN_D08_SPLIT_EXTERNAL", "WARN", "D08_active has no split column; dim_school_split and membership registries carry split", "Do not infer split from row order", "Use school_uid merge to dim_school_split")
if key_audit["pct_range_violation_n"].fillna(0).sum() > 0:
    add_issue("FAIL_PCT_RANGE", "FAIL", "pct range violation found", "Rate interpretation blocked", "Inspect domain anomalies")
else:
    add_issue("PASS_PCT_RANGE", "PASS", "0 pct range violations in active/clean/strict marts", "Non-blocking", "No decision")
if int(d08["outcome_row_id"].duplicated().sum()) == 0:
    add_issue("PASS_OUTCOME_ROW_ID", "PASS", "outcome_row_id duplicate rows = 0", "Stable row ID usable", "No decision")
if not clean.empty and "GRADE_COUNT_READY" in clean.columns:
    for family in ["GRADE_COUNT_READY", "RETENTION_COUNT_READY", "PROGRESSION_COUNT_READY"]:
        if family in clean.columns and str(clean[family].dropna().astype(str).str.lower().unique().tolist()) in {"['not_observed']", "[]"}:
            add_issue(f"WARN_{family}", "WARN", f"{family}=not_observed", "Count-binomial analysis unavailable from current D08", "Obtain numerator/denominator source")
add_issue("WARN_DIRECT_URL_LIMIT", "WARN", "Some source direct download URLs not in manifest; portal URL + local hash retained", "External provenance is portal-level for those files", "Keep local hashes as provenance anchor")
if p6_status.shape[0]:
    q_branch = p6_status.loc[p6_status["status_key"].eq("P6_Q_BRANCH_STATUS"), "status"]
    if len(q_branch) and str(q_branch.iloc[0]) != "READY":
        add_issue("WARN_Q_BRANCH_BLOCKED", "WARN", f"P6_Q_BRANCH_STATUS={q_branch.iloc[0]}", "Selectivity/Q branch is not launch-ready", "Resolve feature contract before Q branch interpretation")
if p5_status_json.get("P5_RESIDUAL_STATUS") == "PENDING_UPSTREAM_RESIDUAL" and p6_status.shape[0]:
    add_issue("WARN_PHASE_STATUS_MIXED", "WARN", "P5 v2 strict reports residual pending, P6 run-up reports residual handoff ready", "Use newest phase-specific artifact when citing residual status", "Freeze canonical phase status if publishing")
issue_register = pd.DataFrame(issue_rows)
atomic_csv(issue_register, QA_DIR / "issue_register.csv")

status_rows = [
    {"stage": "V00_inventory", "status": "PASS" if notebook_inventory["clone_hash_match"].all() else "WARN", "evidence": "representative_notebook_clone_inventory.csv"},
    {"stage": "V01_source_lineage", "status": "PASS" if (PROJECT_ROOT / "workbook/p2/p2_4/p4_preprocessing_integrity_v1/qa/data_source_catalog.csv").exists() else "WARN", "evidence": "data_source_catalog.csv"},
    {"stage": "V02_integrity", "status": "PASS" if int(d08["outcome_row_id"].duplicated().sum()) == 0 and pct_range_violations(d08) == 0 else "FAIL", "evidence": "key_integrity_audit.csv"},
    {"stage": "V03_samples", "status": "PASS" if not sample_summary.empty else "WARN", "evidence": "phase_sample_summary.csv"},
    {"stage": "V04_existing_results", "status": "PASS" if not p6_numbers.empty and not p5_diff.empty else "WARN", "evidence": "existing_result_numeric_readout.csv"},
    {"stage": "V05_interpretation_limits", "status": "PASS", "evidence": "numeric_interpretation_scaffold.csv"},
    {"stage": "Final_handoff", "status": "PASS", "evidence": "INTEGRATED_ENGINEERING_BLUEPRINT_REPORT.md"},
]
status_matrix = pd.DataFrame(status_rows)
atomic_csv(status_matrix, QA_DIR / "blueprint_status_matrix.csv")


# ----------------------------
# Visual readout
# ----------------------------
if plt is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
    fig.suptitle("P2 Integrated Engineering Blueprint - Recomputed Integrity and Existing Result Readout", fontsize=14)

    ax = axes[0, 0]
    if not sample_summary.empty and {"sample_id", "row_n"}.issubset(sample_summary.columns):
        ss = sample_summary[["sample_id", "row_n"]].copy().head(12)
        ax.barh(ss["sample_id"], ss["row_n"], color="#3b6ea8")
        ax.set_title("Phase sample row counts")
        ax.invert_yaxis()
    else:
        ax.text(0.5, 0.5, "No sample registry", ha="center")
    ax.set_xlabel("rows")

    ax = axes[0, 1]
    tp = target_profile[target_profile["dataset"].eq("D08_active")]
    if len(tp):
        ax.bar(tp["target"], tp["n"], color="#6aa84f")
        ax.set_title("D08 target non-null N")
        ax.tick_params(axis="x", rotation=35)
    else:
        ax.text(0.5, 0.5, "No target profile", ha="center")

    ax = axes[1, 0]
    p6_plot_metrics = [
        "P4_RAW_A_EMPLOYMENT_AME_PP",
        "P4_RAW_A_PROGRESSION_AME_PP",
        "P4_RESID_EMPLOYMENT_AME_PP",
        "P4_RESID_PROGRESSION_AME_PP",
    ]
    vals = [p6_metric.get(m, np.nan) for m in p6_plot_metrics]
    ax.bar([m.replace("P4_", "").replace("_AME_PP", "") for m in p6_plot_metrics], vals, color=["#8faadc", "#d9a441", "#8faadc", "#d9a441"])
    ax.set_title("Existing P6 AME readout (pp)")
    ax.tick_params(axis="x", rotation=25)
    ax.axhline(0, color="black", linewidth=0.8)

    ax = axes[1, 1]
    if not p5_diff.empty:
        pp = p5_diff.assign(diff_pp=lambda x: x["difference"].astype(float) * 100).sort_values("diff_pp")
        ax.barh(pp["major_group_7"], pp["diff_pp"], color="#9b7653")
        ax.set_title("P5 progression minus employment AME by major7 (pp)")
        ax.axvline(0, color="black", linewidth=0.8)
    else:
        ax.text(0.5, 0.5, "No P5 difference table", ha="center")
    fig_path = FIG_DIR / "integrated_blueprint_dashboard.png"
    fig.savefig(fig_path, dpi=160)
    plt.close(fig)


# ----------------------------
# Reports and manifest
# ----------------------------
source_catalog_path = PROJECT_ROOT / "workbook/p2/p2_4/p4_preprocessing_integrity_v1/reports/DATA_SOURCE_CATALOG.md"
report_lines = [
    "# P2 Integrated Engineering Blueprint",
    "",
    "## 1. Executive Status",
    "",
    f"- run_id: `{RUN_ID}`",
    f"- output_run_dir: `{RUN_DIR}`",
    f"- final_status: `{'PASS_WITH_WARNINGS' if (issue_register['severity'].eq('WARN').any() and not issue_register['severity'].eq('FAIL').any()) else ('PASS' if not issue_register['severity'].eq('FAIL').any() else 'FAIL')}`",
    f"- representative_notebooks_selected: `{len(notebook_inventory)}`",
    f"- clone_hash_mismatch_n: `{int((~notebook_inventory['clone_hash_match']).sum())}`",
    f"- active D08: `{d08_path.relative_to(PROJECT_ROOT)}` / `{len(d08)} x {len(d08.columns)}` / `{sha256_file(d08_path)}`",
    "",
    "## 2. V-Block Blueprint",
    "",
    md_table(status_matrix, ["stage", "status", "evidence"]),
    "",
    "## 3. Representative Notebook Clones",
    "",
    md_table(notebook_inventory, ["stage", "label", "path", "reason", "cell_n", "code_cell_n", "executed_code_cell_n", "clone_status", "clone_hash_match"]),
    "",
    "## 4. Key File Inventory",
    "",
    md_table(file_inventory, ["label", "role", "path", "exists", "shape", "sha256"], max_rows=60),
    "",
    "## 5. Recomputed Integrity",
    "",
    md_table(key_audit, ["dataset", "shape", "outcome_row_id_null_n", "outcome_row_id_duplicate_rows", "exact_duplicate_rows", "school_n", "campus_n", "dept_n", "major7_non_null_n", "major7_missing_n", "pct_range_violation_n"]),
    "",
    "## 6. Sample Registry",
    "",
    md_table(sample_summary, [c for c in ["sample_id", "row_n", "school_n", "train_n", "validation_n", "test_n"] if c in sample_summary.columns], max_rows=30) if not sample_summary.empty else "_sample registry missing_",
    "",
    "## 7. Target Profile",
    "",
    md_table(target_profile, ["dataset", "target", "n", "missing_n", "mean", "std", "min", "median", "max", "zero_n", "hundred_n"]),
    "",
    "## 8. Existing Numeric Results Readout",
    "",
    md_table(result_summary, ["source", "metric", "value", "unit"], max_rows=50),
    "",
    "## 9. Interpretation Scaffold",
    "",
    md_table(interpretation, ["topic", "observation", "interpretation", "limit"]),
    "",
    "## 10. Issue Register",
    "",
    md_table(issue_register, ["issue_id", "severity", "evidence", "impact", "required_decision"]),
    "",
    "## 11. Data Source Catalog Pointer",
    "",
    f"- source catalog report: `{source_catalog_path.relative_to(PROJECT_ROOT)}`",
    f"- source catalog exists: `{source_catalog_path.exists()}`",
    "",
    "## 12. Rerun Contract",
    "",
    "```bash",
    "MPLCONFIGDIR=/tmp/mplconfig MPLBACKEND=Agg .venv/bin/jupyter nbconvert --to notebook --execute workbook/p2/p2_integrated_engineering_blueprint_v1/P2_INTEGRATED_ENGINEERING_BLUEPRINT_ONE_CELL.ipynb --inplace --ExecutePreprocessor.timeout=900",
    "```",
    "",
    "No new model fitting is performed in this integrated notebook. It clones notebooks, recomputes integrity, reads existing result artifacts, and writes a reproducible handoff bundle.",
]
report_path = REPORT_DIR / "INTEGRATED_ENGINEERING_BLUEPRINT_REPORT.md"
atomic_text("\n".join(report_lines) + "\n", report_path)

handoff_lines = [
    "# HANDOFF TO CHATGPT - P2 Integrated Engineering Blueprint",
    "",
    f"FINAL_STATUS: {'PASS_WITH_WARNINGS' if (issue_register['severity'].eq('WARN').any() and not issue_register['severity'].eq('FAIL').any()) else ('PASS' if not issue_register['severity'].eq('FAIL').any() else 'FAIL')}",
    f"RUN_ID: {RUN_ID}",
    f"OUTPUT_RUN_DIR: {RUN_DIR}",
    f"ACTIVE_D08: {d08_path}",
    f"ACTIVE_D08_SHAPE: {len(d08)} x {len(d08.columns)}",
    f"ACTIVE_D08_SHA256: {sha256_file(d08_path)}",
    f"REPRESENTATIVE_NOTEBOOKS: {len(notebook_inventory)} selected, {int((~notebook_inventory['clone_hash_match']).sum())} clone hash mismatches",
    f"KEY_DUPLICATE: outcome_row_id duplicate rows = {int(d08['outcome_row_id'].duplicated().sum())}",
    f"PCT_RANGE_VIOLATION: {pct_range_violations(d08)}",
    f"TARGET_NON_NULL: {json.dumps({t: int(d08[t].notna().sum()) for t in target_cols}, ensure_ascii=False)}",
    f"P6_OVERALL_STATUS: {p6_status.loc[p6_status['status_key'].eq('P6_OVERALL_STATUS'), 'status'].iloc[0] if len(p6_status.loc[p6_status['status_key'].eq('P6_OVERALL_STATUS')]) else 'UNKNOWN'}",
    f"P5_OVERALL_STATUS: {p5_status_json.get('P5_OVERALL_STATUS', 'UNKNOWN')}",
    "",
    "GATE_MATRIX:",
    md_table(status_matrix, ["stage", "status", "evidence"]),
    "",
    "ISSUES:",
    md_table(issue_register, ["issue_id", "severity", "evidence", "impact", "required_decision"]),
    "",
    "KEY_OUTPUTS:",
    f"- {report_path}",
    f"- {QA_DIR / 'representative_notebook_clone_inventory.csv'}",
    f"- {QA_DIR / 'key_integrity_audit.csv'}",
    f"- {QA_DIR / 'existing_result_numeric_readout.csv'}",
    f"- {FIG_DIR / 'integrated_blueprint_dashboard.png'}",
]
handoff_path = RUN_DIR / "HANDOFF_TO_CHATGPT.md"
atomic_text("\n".join(handoff_lines) + "\n", handoff_path)

generated_files = []
for base in [QA_DIR, REPORT_DIR, LOG_DIR, FIG_DIR, ART_DIR]:
    generated_files.extend([p for p in base.rglob("*") if p.is_file()])
generated_files.append(handoff_path)
generated_files.extend([p for p in CLONE_DIR.rglob("*.ipynb") if p.is_file()])
manifest_rows = []
for p in sorted(set(generated_files)):
    shape, loader, err = shape_repr(p)
    manifest_rows.append(
        {
            "path": str(p),
            "relative_path": str(p.relative_to(PROJECT_ROOT)),
            "size_bytes": p.stat().st_size,
            "shape": shape,
            "loader": loader,
            "load_error": err,
            "sha256": sha256_file(p),
        }
    )
manifest = pd.DataFrame(manifest_rows)
atomic_csv(manifest, RUN_DIR / "INTEGRATED_BLUEPRINT_MANIFEST.csv")
atomic_text(json.dumps(manifest_rows, ensure_ascii=False, indent=2), RUN_DIR / "INTEGRATED_BLUEPRINT_MANIFEST.json")

print("=" * 100)
print("P2 INTEGRATED ENGINEERING BLUEPRINT COMPLETE")
print("=" * 100)
print(f"RUN_ID: {RUN_ID}")
print(f"OUTPUT_RUN_DIR: {RUN_DIR}")
print(f"ACTIVE_D08_SHAPE: {len(d08)} x {len(d08.columns)}")
print(f"ACTIVE_D08_SHA256: {sha256_file(d08_path)}")
print(f"NOTEBOOK_CLONES: {len(notebook_inventory)}; clone_hash_mismatch_n={int((~notebook_inventory['clone_hash_match']).sum())}")
print(f"KEY_DUPLICATE_OUTCOME_ROW_ID: {int(d08['outcome_row_id'].duplicated().sum())}")
print(f"PCT_RANGE_VIOLATION_N: {pct_range_violations(d08)}")
print("TARGET_NON_NULL:", {t: int(d08[t].notna().sum()) for t in target_cols})
print("STATUS_MATRIX:")
print(status_matrix.to_string(index=False))
print("ISSUE_REGISTER:")
print(issue_register.to_string(index=False))
print("INTERPRETATION:")
print(interpretation.to_string(index=False))
print("REPORT:", report_path)
print("HANDOFF:", handoff_path)
print("MANIFEST:", RUN_DIR / "INTEGRATED_BLUEPRINT_MANIFEST.csv")
if plt is not None:
    print("FIGURE:", FIG_DIR / "integrated_blueprint_dashboard.png")


P2 INTEGRATED ENGINEERING BLUEPRINT COMPLETE
RUN_ID: 20260713T084211Z_5b1a3d5
OUTPUT_RUN_DIR: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/runs/20260713T084211Z_5b1a3d5
ACTIVE_D08_SHAPE: 10242 x 151
ACTIVE_D08_SHA256: 598b68b31b5358dfd23839d4c138cc64838d05876b7791980b376c0453f29962
NOTEBOOK_CLONES: 17; clone_hash_mismatch_n=0
KEY_DUPLICATE_OUTCOME_ROW_ID: 0
PCT_RANGE_VIOLATION_N: 0
TARGET_NON_NULL: {'a_rate_pct': 10242, 'health_employment_rate_pct': 7477, 'graduate_school_progression_rate_pct': 7587}
STATUS_MATRIX:
                    stage status                                    evidence
            V00_inventory   PASS representative_notebook_clone_inventory.csv
       V01_source_lineage   PASS                     data_source_catalog.csv
            V02_integrity   PASS                     key_integrity_audit.csv
              V03_samples   PASS                    phase_sample_summary.csv
     V04_existing_results   PASS         existin